# 01_Loading.ipynb — Data Loading
Loads all CSVs, computes dist_N, segment_id, and saves `data.pkl`.

In [19]:
import numpy as np
import pandas as pd
import pickle


## Assigning Alpha value (See section 3)

In [20]:
ALPHA = {
    "constant":  1, # Constant stream slope
    "upward":    2, # Increasing stream slope
    "downward":  0, # Decreasing stream slope
}

## Stream metadata registry

In [21]:
# (group, label, color, marker, trend)
STREAM_META = [
    ("Siachen",      "sia_1",      "#cb2921",  "o", "downward"),
    ("Siachen",      "sia_2",      "#9467bd",  "o", "constant"),
    ("Siachen",      "sia_3",      "#1932d0",  "o", "downward"),
    ("Siachen",      "sia_4",      "#ff7f0e",  "o", "constant"),
    ("Siachen",      "sia_5",      "#17becf",  "o", "constant"),
    ("Siachen",      "sia_6",      "#2ca02c",  "o", "downward"),
    ("Rongbuk",      "ron_1",      "#035063",  "o", "downward"),
    ("Fountain",     "fou_1",      "#c90df3",  "o", "upward"),
    ("Fountain",     "fou_2",      "#278172",  "o", "upward"),
    ("Fountain",     "fou_3",      "#302781",  "o", "upward"),
    ("Fountain",     "fou_4",      "#752781",  "o", "upward"),
    ("Vonbreen",     "von_1",      "#8a55f3",  "o", "upward"),
    ("Vonbreen",     "von_2",      "#412D2D",  "o", "upward"),
    ("Vonbreen",     "von_3",      "#033E62",  "o", "upward"),
    ("Vonbreen",     "von_4",      "#EE5D5D",  "o", "upward"),
    ("Veteranbreen", "vet_1",      "#7e0202",  "o", "constant"),
]


## File path configuration
Each group entry: `files, length, seg_len, vel_col, elev_col`

In [22]:
GROUP_CONFIG = {
    "Siachen": {
        "files":     ["Stream_csv/sia_1.csv",
                      "Stream_csv/sia_2.csv",
                      "Stream_csv/sia_3.csv",
                      "Stream_csv/sia_4.csv",
                      "Stream_csv/sia_5.csv",
                      "Stream_csv/sia_6.csv",
                      
                      ],
        "Length": [12777.89537553210539, 37590.40172122663635, 
                    13586.16687189555887,46840.74314782994770,
                      48213.32977977869450, 21121.16185051896537 ],
        "seg_len":   2000,
        "vel_col":   "Velocity",
        "elev_col":  "elev",
    },
    "Fountain": {
        "files":     ["Stream_csv/fou_1.csv",
                      "Stream_csv/fou_2.csv",
                      "Stream_csv/fou_3.csv",
                      "Stream_csv/fou_4.csv"],
        "Length": [6077.05692063266088, 5224.30298199478511,
                    3741.90339316260361, 3508.55279700896790],
        "seg_len":   500,
        "vel_col":   "velocity",
        "elev_col":  "elev",
    },
    "Vonbreen": {
        "files":     ["Stream_csv/von_1.csv",
                      "Stream_csv/von_2.csv",
                      "Stream_csv/von_3.csv",
                      "Stream_csv/von_4.csv"],
        "Length": [8707.15926548607786, 9012.78072997488562,
                    6936.18057094319647, 9039.86866123152140],
        "seg_len":   500,
        "vel_col":   "velocity",
        "elev_col":  "elev",
    },
    "Veteranbreen": {
        "files":     ["Stream_csv/vet_1.csv"],
        "Length": [24624.53602059209152],
        "seg_len":   2000,
        "vel_col":   "velocity",
        "elev_col":  "elev",
    },
    "Rongbuk": {
        "files":     ["Stream_csv/ron_1.csv"],
        "Length": [20084.59900788963569],
        "seg_len":   2000,
        "vel_col":   "velocity",
        "elev_col":  "elev_8",
    },
}


## Load CSVs

In [23]:
def load_streams(config):
    dfs = []
    for fpath, max_d in zip(config["files"], config["Length"]):
        df = pd.read_csv(fpath)
        df["dist_N"]     = max_d - df["distance"]
        df["segment_id"] = np.floor(df["dist_N"] / config["seg_len"]).astype(int)
        df.attrs["vel_col"]      = config["vel_col"]
        df.attrs["elev_col"]     = config["elev_col"]
        df.attrs["elev_offset"]  = config.get("elev_offset", 0)
        dfs.append(df)
    return dfs

group_dfs = {g: load_streams(cfg) for g, cfg in GROUP_CONFIG.items()}

for group, dfs in group_dfs.items():
    for i, df in enumerate(dfs):
        print(f"  {group} stream {i+1}: {len(df)} rows, "
              f"{df['segment_id'].nunique()} segments")


  Siachen stream 1: 3647 rows, 7 segments
  Siachen stream 2: 10316 rows, 19 segments
  Siachen stream 3: 3855 rows, 7 segments
  Siachen stream 4: 13211 rows, 24 segments
  Siachen stream 5: 12308 rows, 25 segments
  Siachen stream 6: 6294 rows, 11 segments
  Fountain stream 1: 2637 rows, 13 segments
  Fountain stream 2: 1895 rows, 11 segments
  Fountain stream 3: 1465 rows, 9 segments
  Fountain stream 4: 1400 rows, 8 segments
  Vonbreen stream 1: 2462 rows, 18 segments
  Vonbreen stream 2: 2470 rows, 19 segments
  Vonbreen stream 3: 1838 rows, 14 segments
  Vonbreen stream 4: 2257 rows, 19 segments
  Veteranbreen stream 1: 6603 rows, 13 segments
  Rongbuk stream 1: 5273 rows, 11 segments


## Build stream_registry

In [24]:
group_counters = {g: 0 for g in group_dfs}
stream_registry = []
for group, label, color, marker, trend in STREAM_META:
    idx = group_counters[group]
    df  = group_dfs[group][idx]
    group_counters[group] += 1
    stream_registry.append((df, group, label, color, marker, trend))

print(f"Registry built: {len(stream_registry)} streams")


Registry built: 16 streams


In [25]:
# remove rows with NaN in elevation column for all streams
clean_registry = []

for entry in stream_registry:
    df, group, label, color, marker, trend = entry
    ec       = df.attrs.get("elev_col", "elev")
    before   = len(df)
    df_clean = df.dropna(subset=[ec]).reset_index(drop=True)

    # preserve attrs
    df_clean.attrs = df.attrs

    after = len(df_clean)
    print(f"{label:18s}  rows before={before}  rows after={after}  removed={before - after}")

    clean_registry.append((df_clean, group, label, color, marker, trend))

# replace original registry
stream_registry = clean_registry

print("\nNaN rows removed. stream_registry updated.")

sia_1               rows before=3647  rows after=3647  removed=0
sia_2               rows before=10316  rows after=10316  removed=0
sia_3               rows before=3855  rows after=3854  removed=1
sia_4               rows before=13211  rows after=13175  removed=36
sia_5               rows before=12308  rows after=12075  removed=233
sia_6               rows before=6294  rows after=6294  removed=0
ron_1               rows before=5273  rows after=5273  removed=0
fou_1               rows before=2637  rows after=2637  removed=0
fou_2               rows before=1895  rows after=1895  removed=0
fou_3               rows before=1465  rows after=1465  removed=0
fou_4               rows before=1400  rows after=1400  removed=0
von_1               rows before=2462  rows after=2462  removed=0
von_2               rows before=2470  rows after=2470  removed=0
von_3               rows before=1838  rows after=1838  removed=0
von_4               rows before=2257  rows after=2257  removed=0
vet_1           

## Save data.pkl

In [26]:
with open("data.pkl", "wb") as f:
    pickle.dump({"stream_registry": stream_registry, "ALPHA": ALPHA}, f)
print("Saved data.pkl")


Saved data.pkl


#### Print data

In [27]:
with open('data.pkl', 'rb') as file:
    data = pickle.load(file)


print(data)

{'stream_registry': [(      vertex_index      distance       angle      elev            x  \
0                0      0.000000    8.183272  4622.607 -1484712.571   
1                1      3.389307  333.825575  4622.697 -1484712.089   
2                2      7.149581  332.783638  4622.697 -1484715.363   
3                3      9.848806    6.099399  4622.697 -1484715.076   
4                4     12.548031    8.388944  4622.962 -1484714.789   
...            ...           ...         ...       ...          ...   
3642          3642  12764.084179  341.241011  4856.643 -1490690.175   
3643          3643  12768.445372  346.986889  4855.823 -1490691.157   
3644          3644  12772.806565  342.118350  4856.521 -1490692.139   
3645          3645  12775.350970  337.249811  4856.521 -1490693.123   
3646          3646  12777.895376  337.249811  4857.375 -1490694.107   

               y  Velocity  relief        dist_N  segment_id  
0     785310.382   152.319  12.479  1.277790e+04           6  